# Load Dataset

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
DATA_PATH = "/content/drive/MyDrive/IT549_Lab2/movies.csv"
GLOVE_PATH = "/content/drive/MyDrive/IT549_Lab2/wiki_giga_2024_100_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05.050_combined.txt"

# Task:1-Data Preparation

In [5]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

df = df[['overview', 'tagline', 'keywords', 'genres', 'vote_average']]

df = df.dropna().reset_index(drop=True)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (3759, 5)


,overview,tagline,keywords,genres,vote_average
0,"In the 22nd century, a paraplegic Marine is di...",Enter the World of Pandora.,culture clash future space war space colony so...,Action Adventure Fantasy Science Fiction,7.2
1,"Captain Barbossa, long believed to be dead, ha...","At the end of the world, the adventure begins.",ocean drug abuse exotic island east india trad...,Adventure Fantasy Action,6.9
2,A cryptic message from Bond’s past sends him o...,A Plan No One Escapes,spy based on novel secret agent sequel mi6,Action Adventure Crime,6.3
3,Following the death of District Attorney Harve...,The Legend Ends,dc comics crime fighter terrorist secret ident...,Action Crime Drama Thriller,7.6
4,"John Carter is a war-weary, former military ca...","Lost in our world, found in another.",based on novel mars medallion space travel pri...,Action Adventure Science Fiction,6.1


In [10]:
import re
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [11]:
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'\d+', '', text)

    text = text.translate(str.maketrans('', '', string.punctuation))

    tokens = word_tokenize(text)

    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return tokens

In [12]:
df['overview_tokens'] = df['overview'].apply(preprocess)
df['tagline_tokens'] = df['tagline'].apply(preprocess)
df['keywords_tokens'] = df['keywords'].apply(preprocess)

In [13]:
df[['overview_tokens', 'tagline_tokens', 'keywords_tokens']].head()

,overview_tokens,tagline_tokens,keywords_tokens
0,"[in, the, nd, century, a, paraplegic, marine, ...","[enter, the, world, of, pandora]","[culture, clash, future, space, war, space, co..."
1,"[captain, barbossa, long, believed, to, be, de...","[at, the, end, of, the, world, the, adventure,...","[ocean, drug, abuse, exotic, island, east, ind..."
2,"[a, cryptic, message, from, bond, ’, s, past, ...","[a, plan, no, one, escape]","[spy, based, on, novel, secret, agent, sequel,..."
3,"[following, the, death, of, district, attorney...","[the, legend, end]","[dc, comic, crime, fighter, terrorist, secret,..."
4,"[john, carter, is, a, warweary, former, milita...","[lost, in, our, world, found, in, another]","[based, on, novel, mar, medallion, space, trav..."


In [14]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

Train size: 2631
Validation size: 564
Test size: 564


# Task 2 - GloVe Embedding Pipeline

Load GloVe (100D)

In [ ]:
import numpy as np

embedding_dim = 100

glove_dict = {}

with open(GLOVE_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        values = line.split()
        if len(values) == 0:
            continue
        word = values[0]
        try:
            vector = np.asarray(values[1:], dtype='float32')
            glove_dict[word] = vector
        except ValueError:
            print(f"Skipping malformed line: {line.strip()} (word: '{word}', vector_part: '{values[1:]}')")
            continue

print("Total GloVe words loaded:", len(glove_dict))

In [17]:
all_tokens = set()

for col in ['overview_tokens', 'tagline_tokens', 'keywords_tokens']:
    for tokens in df[col]:
        all_tokens.update(tokens)

print("Total unique dataset tokens:", len(all_tokens))

Total unique dataset tokens: 19187


In [18]:
covered_tokens = [word for word in all_tokens if word in glove_dict]

coverage = len(covered_tokens) / len(all_tokens)

print("Embedding Coverage: {:.2%}".format(coverage))

Embedding Coverage: 91.86%


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

def build_tfidf_glove_embeddings(train_tokens, val_tokens, test_tokens):

    train_texts = train_tokens.apply(lambda x: " ".join(x))
    val_texts = val_tokens.apply(lambda x: " ".join(x))
    test_texts = test_tokens.apply(lambda x: " ".join(x))

    vectorizer = TfidfVectorizer()
    vectorizer.fit(train_texts)

    feature_names = vectorizer.get_feature_names_out()
    vocab_dict = {word: idx for idx, word in enumerate(feature_names)}

    def get_doc_embedding(texts):
        tfidf_matrix = vectorizer.transform(texts)
        doc_embeddings = []

        for i in range(tfidf_matrix.shape[0]):
            row = tfidf_matrix[i]
            indices = row.indices
            weights = row.data

            doc_vector = np.zeros(embedding_dim)
            weight_sum = 0

            for idx, weight in zip(indices, weights):
                word = feature_names[idx]

                if word in glove_dict:
                    doc_vector += weight * glove_dict[word]
                    weight_sum += weight

            if weight_sum != 0:
                doc_vector = doc_vector / weight_sum

            doc_embeddings.append(doc_vector)

        return np.array(doc_embeddings)

    X_train = get_doc_embedding(train_texts)
    X_val = get_doc_embedding(val_texts)
    X_test = get_doc_embedding(test_texts)

    return X_train, X_val, X_test

In [20]:
X_train_overview, X_val_overview, X_test_overview = build_tfidf_glove_embeddings(
    train_df['overview_tokens'],
    val_df['overview_tokens'],
    test_df['overview_tokens']
)

print("Shape:", X_train_overview.shape)

Shape: (2631, 100)


# Task 3 - Model A: Rating Prediction (Regression)

In [21]:
X_train_overview, X_val_overview, X_test_overview = build_tfidf_glove_embeddings(
    train_df['overview_tokens'],
    val_df['overview_tokens'],
    test_df['overview_tokens']
)

In [51]:
X_train_tagline, X_val_tagline, X_test_tagline = build_tfidf_glove_embeddings(
    train_df['tagline_tokens'],
    val_df['tagline_tokens'],
    test_df['tagline_tokens']
)

In [23]:
y_train = train_df['vote_average'].values
y_val = val_df['vote_average'].values
y_test = test_df['vote_average'].values

In [24]:
from sklearn.metrics import mean_squared_error
import numpy as np

mean_rating = y_train.mean()

baseline_preds = np.full_like(y_test, mean_rating)

baseline_mse = mean_squared_error(y_test, baseline_preds)
baseline_rmse = np.sqrt(baseline_mse)

print("Baseline MSE:", baseline_mse)
print("Baseline RMSE:", baseline_rmse)

Baseline MSE: 0.8976762200816473
Baseline RMSE: 0.9474577669118804


In [52]:
import torch

X_train_tensor = torch.tensor(X_train_overview, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_overview, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_overview, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1,1)


X_train_tensor_tag = torch.tensor(X_train_tagline, dtype=torch.float32)
X_val_tensor_tag = torch.tensor(X_val_tagline, dtype=torch.float32)
X_test_tensor_tag = torch.tensor(X_test_tagline, dtype=torch.float32)

y_train_tensor_tag = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_val_tensor_tag = torch.tensor(y_val, dtype=torch.float32).view(-1,1)
y_test_tensor_tag = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

In [27]:
import torch.nn as nn

class RegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x)

In [29]:
model = RegressionModel(input_dim=100)

In [30]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [31]:
epochs = 20

for epoch in range(epochs):

    model.train()
    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    loss.backward()
    optimizer.step()

    # Validation loss
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {loss.item():.4f} | Val Loss: {val_loss.item():.4f}")

Epoch 1/20 | Train Loss: 39.9340 | Val Loss: 39.2889
Epoch 2/20 | Train Loss: 39.2787 | Val Loss: 38.5747
Epoch 3/20 | Train Loss: 38.6080 | Val Loss: 37.8230
Epoch 4/20 | Train Loss: 37.9051 | Val Loss: 37.0394
Epoch 5/20 | Train Loss: 37.1724 | Val Loss: 36.2361
Epoch 6/20 | Train Loss: 36.4153 | Val Loss: 35.4035
Epoch 7/20 | Train Loss: 35.6269 | Val Loss: 34.5255
Epoch 8/20 | Train Loss: 34.7939 | Val Loss: 33.5949
Epoch 9/20 | Train Loss: 33.9078 | Val Loss: 32.6049
Epoch 10/20 | Train Loss: 32.9641 | Val Loss: 31.5513
Epoch 11/20 | Train Loss: 31.9587 | Val Loss: 30.4302
Epoch 12/20 | Train Loss: 30.8880 | Val Loss: 29.2395
Epoch 13/20 | Train Loss: 29.7496 | Val Loss: 27.9788
Epoch 14/20 | Train Loss: 28.5425 | Val Loss: 26.6459
Epoch 15/20 | Train Loss: 27.2640 | Val Loss: 25.2358
Epoch 16/20 | Train Loss: 25.9092 | Val Loss: 23.7435
Epoch 17/20 | Train Loss: 24.4739 | Val Loss: 22.1734
Epoch 18/20 | Train Loss: 22.9601 | Val Loss: 20.5323
Epoch 19/20 | Train Loss: 21.3734 | V

In [32]:
model.eval()

with torch.no_grad():
    test_preds = model(X_test_tensor)

test_preds = test_preds.numpy()
y_test_np = y_test_tensor.numpy()

mse = mean_squared_error(y_test_np, test_preds)
rmse = np.sqrt(mse)

print("Neural Model MSE:", mse)
print("Neural Model RMSE:", rmse)

Neural Model MSE: 16.742835998535156
Neural Model RMSE: 4.091801070254412


In [53]:
model_tag = RegressionModel(input_dim=100)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_tag.parameters(), lr=0.001)

In [54]:
for epoch in range(20):

    model_tag.train()
    optimizer.zero_grad()

    outputs = model_tag(X_train_tensor_tag)
    loss = criterion(outputs, y_train_tensor_tag)

    loss.backward()
    optimizer.step()

In [55]:
model_tag.eval()

with torch.no_grad():
    preds_tag = model_tag(X_test_tensor_tag)

preds_tag = preds_tag.numpy()

mse_tag = mean_squared_error(y_test, preds_tag)
rmse_tag = np.sqrt(mse_tag)

print("Tagline RMSE:", rmse_tag)

Tagline RMSE: 4.478024468365189


In [57]:
print("Overview RMSE:", rmse)
print("Tagline RMSE:", rmse_tag)

Overview RMSE: 4.091801070254412
Tagline RMSE: 4.478024468365189


In [59]:
results = {
    "Overview": rmse,
    "Tagline": rmse_tag
}

results

{'Overview': np.float64(4.091801070254412),
 'Tagline': np.float64(4.478024468365189)}

# Task 4 - Model B: Genre Prediction (Multi-Label Classification)

In [34]:
from sklearn.preprocessing import MultiLabelBinarizer

df['genre_list'] = df['genres'].apply(lambda x: [g.strip() for g in x.split(',')])

mlb = MultiLabelBinarizer()

genre_encoded = mlb.fit_transform(df['genre_list'])

num_classes = len(mlb.classes_)

print("Number of genres:", num_classes)
print("Genres:", mlb.classes_)

Number of genres: 1024
Genres: ['Action' 'Action Adventure' 'Action Adventure Animation Comedy Family'
 ... 'Western Drama Adventure Thriller' 'Western History'
 'Western History War']


In [35]:
train_df['genre_encoded'] = list(genre_encoded[train_df.index])
val_df['genre_encoded'] = list(genre_encoded[val_df.index])
test_df['genre_encoded'] = list(genre_encoded[test_df.index])

In [100]:
import torch.nn as nn

class GenreClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.model(x)

In [101]:
from sklearn.metrics import f1_score, hamming_loss, jaccard_score
import torch

def run_multilabel_experiment(column_name):


    X_train, X_val, X_test = build_tfidf_glove_embeddings(
        train_df[column_name],
        val_df[column_name],
        test_df[column_name]
    )

    y_train = np.array(list(train_df['genre_encoded']))
    y_val = np.array(list(val_df['genre_encoded']))
    y_test = np.array(list(test_df['genre_encoded']))

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
    y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

    model = GenreClassifier(input_dim=100, num_classes=num_classes)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    epochs = 20

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        outputs = model(X_train_tensor)
        loss = criterion(outputs, y_train_tensor)

        loss.backward()
        optimizer.step()

    # Evaluation
    model.eval()
    with torch.no_grad():
        logits = model(X_test_tensor)
        probs = torch.sigmoid(logits)

    # Convert probabilities to binary predictions
    preds = (probs > 0.15).int().numpy()
    y_test_np = y_test_tensor.numpy()

    # Metrics
    micro_f1 = f1_score(y_test_np, preds, average='micro',zero_division=0)
    macro_f1 = f1_score(y_test_np, preds, average='macro',zero_division=0)
    hamming = hamming_loss(y_test_np, preds)
    jaccard = jaccard_score(y_test_np, preds, average='micro')

    return micro_f1, macro_f1, hamming, jaccard

In [102]:
micro_o, macro_o, ham_o, jac_o = run_multilabel_experiment('overview_tokens')

print("Overview Results")
print("Micro-F1:", micro_o)
print("Macro-F1:", macro_o)
print("Hamming Loss:", ham_o)
print("Jaccard Score:", jac_o)

Overview Results
Micro-F1: 0.002108860141567
Macro-F1: 0.0007770996995544681
Hamming Loss: 0.17697424922429078
Jaccard Score: 0.0010555430671344939


In [103]:
micro_t, macro_t, ham_t, jac_t = run_multilabel_experiment('tagline_tokens')

print("Tagline Results")
print("Micro-F1:", micro_t)
print("Macro-F1:", macro_t)
print("Hamming Loss:", ham_t)
print("Jaccard Score:", jac_t)

Tagline Results
Micro-F1: 0.0022172949002217295
Macro-F1: 0.0012906405689373397
Hamming Loss: 0.043633643617021274
Jaccard Score: 0.0011098779134295228


In [65]:
results = {
    "Overview": [micro_o, macro_o, ham_o, jac_o],
    "Tagline": [micro_t, macro_t, ham_t, jac_t]
}

results

{'Overview': [0.0018601463641481264,
  0.0007820858632937822,
  0.21183614527925532,
  np.float64(0.0009309390234939611)],
 'Tagline': [0.0018339706381300836,
  0.0005416901740741729,
  0.18847829399379432,
  np.float64(0.0009178269529062991)]}

# Task 5 - Frequent Words per Genre

In [68]:
from collections import Counter
import pandas as pd
df['genre_list'] = df['genres'].apply(lambda x: [g.strip() for g in x.split(',')])

In [69]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [76]:
def analyze_genre_words_clean(df, text_column='overview_tokens', min_freq=3):

    results = []

    all_genres = sorted(set(g for sublist in df['genre_list'] for g in sublist))

    for genre in all_genres:

        genre_movies = df[df['genre_list'].apply(lambda x: genre in x)]

        all_words = []

        for tokens in genre_movies[text_column]:
            filtered = [
                w for w in tokens
                if w not in stop_words
                and len(w) > 3
                and w.isalpha()
            ]
            all_words.extend(filtered)

        word_counts = Counter(all_words)

        top_10 = word_counts.most_common(10)

        valid_words = [(w, c) for w, c in word_counts.items() if c >= min_freq]
        valid_words_sorted = sorted(valid_words, key=lambda x: x[1])
        bottom_10 = valid_words_sorted[:10]

        for word, count in top_10:
            results.append([genre, "Top", word, count])

        for word, count in bottom_10:
            results.append([genre, "Bottom", word, count])

    result_df = pd.DataFrame(results, columns=["Genre", "Type", "Word", "Frequency"])

    return result_df

In [77]:
df['genre_list'] = df['genres'].apply(lambda x: [g.strip() for g in x.split(',')])

freq_table = analyze_genre_words_clean(df, text_column='overview_tokens')

freq_table.head(20)

,Genre,Type,Word,Frequency
0,Action,Top,brother,5
1,Action,Top,must,5
2,Action,Top,terry,5
3,Action,Top,gang,4
4,Action,Top,agent,4
5,Action,Top,star,3
6,Action,Top,fellow,3
7,Action,Top,take,3
8,Action,Top,fighter,3
9,Action,Top,biker,3


In [78]:
for genre in freq_table['Genre'].unique():

    print("\n==============================")
    print(f"Genre: {genre}")

    print("\nTop 10 Frequent Words:")
    print(freq_table[(freq_table['Genre'] == genre) &
                     (freq_table['Type'] == 'Top')]
          [['Word', 'Frequency']]
          .reset_index(drop=True))

    print("\nBottom 10 Least Frequent Words (freq ≥ 3):")
    print(freq_table[(freq_table['Genre'] == genre) &
                     (freq_table['Type'] == 'Bottom')]
          [['Word', 'Frequency']]
          .reset_index(drop=True))

Streaming output truncated to the last 5000 lines.
0   alien          3
1    crew          3
2  planet          4

Genre: Horror Comedy

Top 10 Frequent Words:
       Word  Frequency
0   vampire          5
1      life          4
2     becca          3
3      town          3
4  werewolf          2
5    change          2
6     young          2
7      must          2
8   suspect          2
9    anyone          2

Bottom 10 Least Frequent Words (freq ≥ 3):
      Word  Frequency
0    becca          3
1     town          3
2     life          4
3  vampire          5

Genre: Horror Comedy Action Science Fiction Thriller

Top 10 Frequent Words:
             Word  Frequency
0           eaten          1
1           alive          1
2         unknown          1
3        creature          1
4           local          1
5            game          1
6          warden          1
7            team          1
8  paleontologist          1
9            york          1

Bottom 10 Least Frequent Words (fre

# Task 6 - Genre-Indicative Words Using TF-IDF

In [79]:
df['overview_clean'] = df['overview_tokens'].apply(lambda x: " ".join(x))

In [80]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    min_df=3
)

X = vectorizer.fit_transform(df['overview_clean'])

In [82]:
from sklearn.preprocessing import MultiLabelBinarizer

df['genre_list'] = df['genres'].apply(lambda x: [g.strip() for g in x.split(',')])

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(df['genre_list'])

genres = mlb.classes_

In [83]:
from sklearn.linear_model import LogisticRegression
import numpy as np

indicative_words = {}

feature_names = vectorizer.get_feature_names_out()

for i, genre in enumerate(genres):

    y = Y[:, i]

    if y.sum() < 10:
        continue

    clf = LogisticRegression(max_iter=2000)
    clf.fit(X, y)

    coef = clf.coef_[0]

    top_indices = np.argsort(coef)[-10:]

    top_words = [(feature_names[j], coef[j]) for j in reversed(top_indices)]

    indicative_words[genre] = top_words

In [84]:
for genre, words in indicative_words.items():

    print("\n==============================")
    print(f"Genre: {genre}")
    print("Top 10 Indicative Words:")

    for word, weight in words:
        print(f"{word:15} {weight:.4f}")


Genre: Action
Top 10 Indicative Words:
fighter         0.8650
brother         0.8045
terry           0.6763
bank            0.6052
fellow          0.5722
bring           0.5371
agent           0.5352
biker           0.5268
star            0.4915
toretto         0.4793

Genre: Action Adventure Comedy
Top 10 Indicative Words:
criminal        0.7277
mitchell        0.6294
protection      0.5733
bikers          0.5559
cousin          0.5490
thief           0.5482
delivery        0.5451
jackson         0.5016
obtain          0.5007
evil            0.4622

Genre: Action Adventure Drama Thriller
Top 10 Indicative Words:
prison          0.6172
thief           0.5548
sam             0.5221
copilot         0.5046
champion        0.4916
terrorist       0.4271
alive           0.4268
heavyweight     0.4045
plane           0.4042
advertisement   0.3979

Genre: Action Adventure Science Fiction
Top 10 Indicative Words:
earth           1.0178
world           0.9921
alien           0.8716
captain      